# InsightMem Quickstart

目标：离线完成一次 Add/Search，验证幂等重放与用户隔离，并把结果保存到独立运行目录。

这个 Notebook 使用确定性 `HashEmbedder`，不需要 API Key 或网络。核心检索逻辑仍来自仓库根目录的 `insightmem` 包。

In [ ]:
import json

from insightmem.coding.config import CodingConfig
from insightmem.store import MemoryMessage
from insightmem_lab import (
    HashEmbedder,
    find_project_root,
    load_sample_messages,
    start_experiment,
)

PROJECT_ROOT = find_project_root()
REPO_ROOT = PROJECT_ROOT.parents[1]
config = CodingConfig.load(REPO_ROOT / "configs" / "coding" / "default.json")

print(f"Project: {PROJECT_ROOT}")
print(f"Config fingerprint input: {config}")

## 1. 创建一次独立运行

每次调用都会创建新的 `runs/<run_id>/memory.sqlite3` 和 `manifest.json`。修改 embedding 或配置时不需要覆盖旧实验。

In [ ]:
run = start_experiment(
    PROJECT_ROOT,
    "quickstart",
    config=config,
    embedder=HashEmbedder(),
)

print(f"Run: {run.run_id}")
print(f"Directory: {run.directory}")

## 2. Add：写入样本并检查幂等

同一个 `(user_id, request_id)` 和同一 payload 重放时，不应产生重复 source。

In [ ]:
messages = load_sample_messages(PROJECT_ROOT)
stored_chunks = run.add("request-demo-1", "user-a", "session-demo", messages)
replayed_chunks = run.add("request-demo-1", "user-a", "session-demo", messages)

print(f"Stored chunks: {stored_chunks}")
print(f"Replay chunks: {replayed_chunks}")
assert stored_chunks == replayed_chunks

## 3. Search：查看有序证据

结果用于观察检索与选择，不生成最终答案。保存原始 hits，后续可在 L7 中计算指标和成本。

In [ ]:
query = "Why does the parser drop backslashes?"
hits = run.search("user-a", query, top_k=5)

print(json.dumps(hits, ensure_ascii=False, indent=2))
assert hits

## 4. 双用户哨兵：检查隔离

用户 B 写入独有短语；用户 A 不应看到该内容。这个检查与同用户噪声实验不同。

In [ ]:
sentinel = "B-ONLY-SENTINEL"
run.add(
    "request-demo-b",
    "user-b",
    "session-demo-b",
    [MemoryMessage(role="user", content=f"{sentinel} belongs only to user B.")],
)

user_a_hits = run.search("user-a", sentinel, top_k=10)
user_b_hits = run.search("user-b", sentinel, top_k=10)
leaked = any(sentinel in hit["content"] for hit in user_a_hits)
visible_to_b = any(sentinel in hit["content"] for hit in user_b_hits)

print(f"Leaked to user A: {leaked}")
print(f"Visible to user B: {visible_to_b}")
assert not leaked
assert visible_to_b

## 5. 保存原始产物

`manifest.json` 已由实验接口保存；这里补充 query、hits、隔离检查和统计信息。

In [ ]:
result_path = run.save(
    "quickstart-results.json",
    {
        "query": query,
        "stored_chunks": stored_chunks,
        "replayed_chunks": replayed_chunks,
        "hits": hits,
        "isolation": {
            "sentinel": sentinel,
            "leaked_to_user_a": leaked,
            "visible_to_user_b": visible_to_b,
        },
        "stats": run.stats(),
    },
)

print(f"Results: {result_path}")
print(f"Manifest: {run.directory / 'manifest.json'}")
print(f"Stats: {run.stats()}")

## 下一步

复制这个 Notebook，在 `src/insightmem_lab/` 中添加可复用实验函数，再保持每个 Notebook 只做四件事：设置变量、调用实验、展示结果、保存产物。

可参考的学习实验：L1/L2 索引对照、L3 三路召回消融、L4 噪声配对、L5 去重与合并、L7 评测报告。L3/L5 需要实验分支或离线调用内部函数，当前公开 Add/Search 没有逐腿或 dedup 开关。